### Testing RAG Applications 📑

#### RAG Application
This application reads data about Model Context Protocol (MCP) server from internet, stores in vector stores, chunks the data with embedding and useful to answer the question about MCP while inferenced.

<img src="./img/RAG.png" width="1000" height="800" style="display: block; margin: auto;">

In [ ]:
# %pip install -qU langchain langchain-community langchain-chroma langchain-ollama langchain-text-splitters
%pip install beautifulsoup4




In [5]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_ollama import ChatOllama

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:latest",
    temperature=0.5,
    max_tokens = 250
)

In [7]:
# Load data from Web
loader = WebBaseLoader("https://www.descope.com/learn/post/mcp")
data = loader.load() # beautifulsoup4 is required for web loading.

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever()

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}
    
    Give a summary not the full detail

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()


- Load data from the web
    - WebBaseLoader fetches the HTML content from the given URL.
    - It uses BeautifulSoup (bs4) internally to parse the page.
    - data is a list of Document objects containing the text content of the page.
- Embed and store in a vector database
    - Each chunk is converted into a vector (numerical representation) using Ollama’s embedding model (nomic-embed-text).
    - These vectors are stored in a Chroma vector database.
    - This allows semantic search: finding chunks most relevant to a query
- Create a retriever
    - The retriever is an interface to query the vector database.
    - Instead of searching by keywords, it retrieves semantically similar chunks.
- Format retrieved documents
    - Takes a list of Document objects and joins their text content into one string.
    - This string will be fed into the LLM as context.
- Retrieve and format documents
    - For a given question, the retriever finds relevant chunks.
    - Those chunks are formatted into a single string.
- Build the RAG chain
    - This is the most important part:
    - Step 1: A dictionary maps:
        - "context" → the function that retrieves and formats docs.
        - "question" → RunnablePassthrough() (just forwards the user’s question).
    - Step 2: Passes both into the prompt template.
    - Step 3: The filled prompt goes into the LLM (llm = ChatOllama).
    - Step 4: StrOutputParser() extracts the raw text output from the LLM.
    - So the chain is:
        - User Question → Retriever → Prompt → LLM → Parsed Answer



#### Output of the LLM Application

In [ ]:
response = chain.invoke("What is MCP")

print(response)

### Testing RAG Application with DeepEval
<img src="./img/RAGTesting.png" width="800" height="400" style="display: block; margin: auto;">

In [8]:
from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset

test_case = LLMTestCase(
    input="What is MCP?",
    actual_output=chain.invoke("What is MCP"),
    expected_output="The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps"
)

dataset = EvaluationDataset()
dataset.add_test_case(test_case=test_case)

In [9]:
dataset

EvaluationDataset(test_cases=[LLMTestCase(input='What is MCP?', actual_output='MCP, or Model Context Protocol, standardizes how LLMs use function calls by streaming tool definitions to the LLM. This helps connect AI apps to context and builds on existing function calling methods to simplify and standardize development.', expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps', context=None, retrieval_context=None, additional_metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None)], goldens=[], _alias=None, _id=None, _multi_turn=False)

In [10]:
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import GEval

concise_metrics = GEval(
    name = "Concise",
    criteria="Assess if the actual output remains concise while preserving all essential information.",
    
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

In [11]:
from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import GEval

completness_metrics = GEval(
    name = "Completeness",
    criteria="Assess whether the actual output retains all the key information from the input",
    
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT
    ]
)

### Evaluation with GEval 

In [17]:
from deepeval.evaluate import evaluate
from deepeval.metrics import AnswerRelevancyMetric

evaluate(dataset.test_cases, metrics=[
    completness_metrics, 
    AnswerRelevancyMetric(),
    concise_metrics
])

# So in one evaluation run, we are able to check for completeness, conciseness, and answer relevancy.
# And its result are displayed in Confident AI platform.

ImportError: cannot import name 'ConcisenessMetric' from 'deepeval.metrics' (c:\Users\Lenovo\Downloads\aiqaDemoSource\myenv\Lib\site-packages\deepeval\metrics\__init__.py)